# 01 - Bronze Ingestion

## Purpose

Read source CSV files from Lakehouse Files, add ingestion metadata, preserve the raw source structure, and write Bronze Delta tables.

## Bronze Design Rules

- Preserve source columns as received.
- Add metadata columns for lineage and troubleshooting.
- Avoid business transformations in Bronze.
- Log row counts for every entity.

## Output Tables

- bronze_customers
- bronze_accounts
- bronze_products
- bronze_branches
- bronze_transactions

In [ ]:
from datetime import datetime, timezone
from pyspark.sql import functions as F

batch_id = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
source_base_path = "Files/retail_banking/source"
entities = ["customers", "accounts", "products", "branches", "transactions"]

print(f"Bronze ingestion batch: {batch_id}")

In [ ]:
def read_source_csv(entity_name: str):
    path_value = f"{source_base_path}/{entity_name}.csv"
    return (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "false")
        .option("multiLine", "false")
        .load(path_value)
    )


def add_ingestion_metadata(df, entity_name: str):
    return (
        df
        .withColumn("_ingestion_timestamp", F.current_timestamp())
        .withColumn("_source_file_name", F.input_file_name())
        .withColumn("_source_entity", F.lit(entity_name))
        .withColumn("_batch_id", F.lit(batch_id))
    )


def write_bronze(df, entity_name: str):
    table_name = f"bronze_{entity_name}"
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )
    row_count = df.count()
    print(f"{table_name}: {row_count} rows written")
    return row_count

In [ ]:
bronze_counts = []

for entity in entities:
    raw_df = read_source_csv(entity)
    bronze_df = add_ingestion_metadata(raw_df, entity)
    row_count = write_bronze(bronze_df, entity)
    bronze_counts.append((entity, row_count, batch_id))

bronze_count_df = spark.createDataFrame(bronze_counts, ["entity", "row_count", "batch_id"])
display(bronze_count_df)

In [ ]:
# Validate that every expected Bronze table has rows.
for entity in entities:
    table_name = f"bronze_{entity}"
    count_value = spark.table(table_name).count()
    if count_value == 0:
        raise ValueError(f"Bronze table {table_name} has zero rows")
    print(f"Validation passed: {table_name} has {count_value} rows")

## Output Explanation

The count table confirms how many rows were loaded per entity for this batch. In production, write these counts to an audit table so pipelines can compare source and target counts.